In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/pm-81808267-at-04-30-2025-02-25-23/__script__.py
/kaggle/input/pm-81808267-at-04-30-2025-02-25-23/__results__.html
/kaggle/input/pm-81808267-at-04-30-2025-02-25-23/input_requirements.txt
/kaggle/input/pm-81808267-at-04-30-2025-02-25-23/__script__.ipynb
/kaggle/input/pm-81808267-at-04-30-2025-02-25-23/__output__.json
/kaggle/input/pm-81808267-at-04-30-2025-02-25-23/custom.css


Build an ETL Data Pipeline Extracting Weather Data via OpenWeather API (Python/PostgreSQL/SQL): This tutorial demonstrates how to extract weather data using the OpenWeather API, transform it, and load it into a PostgreSQL database. -->

In [2]:
import requests
import json
import pandas as pd #Data manipulation
from datetime import datetime 
import matplotlib.pyplot as plt #Data Viz
import seaborn as sns #Data Viz

In [3]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00


In [4]:
# Set up API session
import openmeteo_requests
import requests_cache
from retry_requests import retry


cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)


# Extract

In [5]:
# Get the weather data from the API 
import openmeteo_requests

import requests_cache
import pandas as pd
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 42.947,
	"longitude": -76.4291,
	"hourly": ["temperature_2m", "relative_humidity_2m", 
               "precipitation_probability", "rain", "showers", "weather_code"],
	"forecast_days": 16,
	"wind_speed_unit": "mph",
	"temperature_unit": "fahrenheit",
	"precipitation_unit": "inch"
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation {response.Elevation()} m asl")
print(f"Timezone {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(2).ValuesAsNumpy()
hourly_rain = hourly.Variables(3).ValuesAsNumpy()
hourly_showers = hourly.Variables(4).ValuesAsNumpy()
hourly_weathercode = hourly.Variables(5).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["rain"] = hourly_rain
hourly_data["showers"] = hourly_showers
hourly_data['weather_code'] = hourly_weathercode

hourly_dataframe = pd.DataFrame(data = hourly_data)
hourly_dataframe

Coordinates 42.93283462524414°N -76.42066192626953°E
Elevation 268.0 m asl
Timezone NoneNone
Timezone difference to GMT+0 0 s


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,date,temperature_2m,relative_humidity_2m,precipitation_probability,rain,showers,weather_code
0,2025-04-30 00:00:00+00:00,67.083801,77.0,67.0,0.023622,0.0,53.0
1,2025-04-30 01:00:00+00:00,66.633797,79.0,29.0,0.000000,0.0,3.0
2,2025-04-30 02:00:00+00:00,66.003799,91.0,47.0,0.354331,0.0,65.0
3,2025-04-30 03:00:00+00:00,53.763802,82.0,57.0,0.023622,0.0,53.0
4,2025-04-30 04:00:00+00:00,49.533798,76.0,7.0,0.000000,0.0,3.0
...,...,...,...,...,...,...,...
379,2025-05-15 19:00:00+00:00,NaN,NaN,30.0,NaN,NaN,NaN
380,2025-05-15 20:00:00+00:00,NaN,NaN,34.0,NaN,NaN,NaN
381,2025-05-15 21:00:00+00:00,NaN,NaN,38.0,NaN,NaN,NaN
382,2025-05-15 22:00:00+00:00,NaN,NaN,41.0,NaN,NaN,NaN


In [6]:
hourly_dataframe.dtypes

date                         datetime64[ns, UTC]
temperature_2m                           float32
relative_humidity_2m                     float32
precipitation_probability                float32
rain                                     float32
showers                                  float32
weather_code                             float32
dtype: object

In [7]:
hourly_dataframe['date'].dt.hour

0       0
1       1
2       2
3       3
4       4
       ..
379    19
380    20
381    21
382    22
383    23
Name: date, Length: 384, dtype: int32

# Transform

Columns/Variables to modify

All columns -> Rename properly 

* Date: Clean up for month
* Temperature: Round up to nearest digit
* Relative Humidity
* Precipitation Probability
* Rain
* Showers
* Weather code: Convert number to description



In [8]:
#Fill NaN for when new hourly data is not yet avaialable in the 16th days
#Using forward fill since we don't have that during the new hourly. We are moving forward in time.
hourly_dataframe['temperature_2m'].ffill(inplace=True)
hourly_dataframe['weather_code'].ffill(inplace=True)
hourly_dataframe['relative_humidity_2m'].ffill(inplace=True)
hourly_dataframe['precipitation_probability'].ffill(inplace=True)
hourly_dataframe['rain'].ffill(inplace=True)
hourly_dataframe['showers'].ffill(inplace=True)
hourly_dataframe

/tmp/ipykernel_14/2769882998.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  hourly_dataframe['temperature_2m'].ffill(inplace=True)
/tmp/ipykernel_14/2769882998.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

,date,temperature_2m,relative_humidity_2m,precipitation_probability,rain,showers,weather_code
0,2025-04-30 00:00:00+00:00,67.083801,77.0,67.0,0.023622,0.0,53.0
1,2025-04-30 01:00:00+00:00,66.633797,79.0,29.0,0.000000,0.0,3.0
2,2025-04-30 02:00:00+00:00,66.003799,91.0,47.0,0.354331,0.0,65.0
3,2025-04-30 03:00:00+00:00,53.763802,82.0,57.0,0.023622,0.0,53.0
4,2025-04-30 04:00:00+00:00,49.533798,76.0,7.0,0.000000,0.0,3.0
...,...,...,...,...,...,...,...
379,2025-05-15 19:00:00+00:00,57.524002,88.0,30.0,0.000000,0.0,3.0
380,2025-05-15 20:00:00+00:00,57.524002,88.0,34.0,0.000000,0.0,3.0
381,2025-05-15 21:00:00+00:00,57.524002,88.0,38.0,0.000000,0.0,3.0
382,2025-05-15 22:00:00+00:00,57.524002,88.0,41.0,0.000000,0.0,3.0


In [9]:
# Clean temperature

#Opt to use fillna instead so I don't encounter this error: 
#ValueError: Length of values (379) does not match length of index (384)

temp_list = hourly_dataframe['temperature_2m'].tolist()
temp_list = [round(x) for x in temp_list if (not np.isnan(x))]

hourly_dataframe['temperature_2m'] = temp_list

In [10]:
# Clean weather code
weather_code_desc = {
                    0: 'Clear Sky',
                    1: 'Mainly clear, partly cloudy, and overcast',
                    2: 'Mainly clear, partly cloudy, and overcast',
                    3: 'Mainly clear, partly cloudy, and overcast',
                    45: 'Fog and depositing rime fog',
                    48: 'Fog and depositing rime fog',
                    51: 'Drizzle: Light, moderate, and dense intensity',
                    53: 'Drizzle: Light, moderate, and dense intensity',
                    55: 'Drizzle: Light, moderate, and dense intensity',
                    56: 'Freezing Drizzle: Light and dense intensity',
                    57: 'Freezing Drizzle: Light and dense intensity',
                    61: 'Rain: Slight, moderate and heavy intensity',
                    63: 'Rain: Slight, moderate and heavy intensity',
                    65: 'Rain: Slight, moderate and heavy intensity',
                    66: 'Freezing Rain: Light and heavy intensity',
                    67: 'Freezing Rain: Light and heavy intensity',
                    71: 'Snow fall: Slight, moderate, and heavy intensity',
                    73: 'Snow fall: Slight, moderate, and heavy intensity',
                    75: 'Snow fall: Slight, moderate, and heavy intensity',
                    77: 'Snow grains',
                    80: 'Rain showers: Slight, moderate, and violent',
                    81: 'Rain showers: Slight, moderate, and violent',
                    82: 'Rain showers: Slight, moderate, and violent',
                    85: 'Snow showers slight and heavy',
                    86: 'Snow showers slight and heavy',
                    95: 'Thunderstorm: Slight or moderate',
                    96: 'Thunderstorm with slight and heavy hail',
                    99: 'Thunderstorm with slight and heavy hail'
}



In [11]:
weather_list = pd.to_numeric(hourly_dataframe['weather_code']).tolist()
weather_list = [ val for key,val in weather_code_desc.items() for x in weather_list if key == x]
hourly_dataframe['weather_desc'] = weather_list
hourly_dataframe['weather_desc']


0                                        Clear Sky
1                                        Clear Sky
2                                        Clear Sky
3                                        Clear Sky
4                                        Clear Sky
                          ...                     
379    Rain showers: Slight, moderate, and violent
380    Rain showers: Slight, moderate, and violent
381    Rain showers: Slight, moderate, and violent
382    Rain showers: Slight, moderate, and violent
383    Rain showers: Slight, moderate, and violent
Name: weather_desc, Length: 384, dtype: object

In [12]:
hourly_dataframe.rename(columns={"temperature_2m": "Temperature (°F)", 
                                 "relative_humidity_2m":"Relative Humidity (%)",
                                 "precipitation_probability" : "Precipitation Probability (%)",
                                 "rain" : "Rain (in)", 
                                 "showers" : "Showers (in)",
                                 "date" : "Datetime",
                                 "weather_code" :"Weather Code"
                                })



,Datetime,Temperature (°F),Relative Humidity (%),Precipitation Probability (%),Rain (in),Showers (in),Weather Code,weather_desc
0,2025-04-30 00:00:00+00:00,67,77.0,67.0,0.023622,0.0,53.0,Clear Sky
1,2025-04-30 01:00:00+00:00,67,79.0,29.0,0.000000,0.0,3.0,Clear Sky
2,2025-04-30 02:00:00+00:00,66,91.0,47.0,0.354331,0.0,65.0,Clear Sky
3,2025-04-30 03:00:00+00:00,54,82.0,57.0,0.023622,0.0,53.0,Clear Sky
4,2025-04-30 04:00:00+00:00,50,76.0,7.0,0.000000,0.0,3.0,Clear Sky
...,...,...,...,...,...,...,...,...
379,2025-05-15 19:00:00+00:00,58,88.0,30.0,0.000000,0.0,3.0,"Rain showers: Slight, moderate, and violent"
380,2025-05-15 20:00:00+00:00,58,88.0,34.0,0.000000,0.0,3.0,"Rain showers: Slight, moderate, and violent"
381,2025-05-15 21:00:00+00:00,58,88.0,38.0,0.000000,0.0,3.0,"Rain showers: Slight, moderate, and violent"
382,2025-05-15 22:00:00+00:00,58,88.0,41.0,0.000000,0.0,3.0,"Rain showers: Slight, moderate, and violent"
